# Dataspace Automated BFF Contract Negotiation & PUSH Transfer Flow

This notebook demonstrates the end-to-end execution of automated BFF contract negotiation followed by PUSH data transfer via Eunomia RPC:
1. Executes prerequisites and automated BFF contract negotiation (`bff_contract_negotiation.ipynb` with `setup_catalog_policies.ipynb`).
2. Initiates PUSH transfer request with destination sink `dataAddress` (`setup-request`).
3. Starts push transfer from Provider (`setup-start`).
4. Suspends transfer from Provider (`setup-suspension`).
5. Resumes transfer from Provider (`setup-start`).
6. Completes transfer from Provider (`setup-completion`).
7. Queries final transfer process state via the Transfer Agent Management API.

## 1. Prerequisites
Execute catalog configuration and automated BFF contract negotiation to obtain an active contract agreement ID.

In [ ]:
%run ./setup_catalog_policies.ipynb
%run ./bff_contract_negotiation.ipynb

## 2. Transfer Request Setup with Sink DataAddress (Consumer -> Provider)
The Consumer initiates the PUSH transfer by sending a `RpcTransferRequestMessage` with format `http+pushing` and destination sink `dataAddress`.
The Provider transitions the process to `REQUESTED` and returns its `providerPid`.

In [ ]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-request"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferRequestMessage",
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+pushing",
    "dataAddress": {
        "endpointType": "https://w3id.org/idsa/v4.1/HTTP",
        "endpoint": "http://127.0.0.1:1112",
        "endpointProperties": [
            {
                "name": "authorization",
                "value": "TOKEN-ABCDEFG"
            },
            {
                "name": "authType",
                "value": "bearer"
            }
        ]
    },
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
transfer_process_consumer_pid = response_as_json["transferAgentModel"]["correlation"]["consumerPid"]
transfer_process_provider_pid = response_as_json["transferAgentModel"]["correlation"]["providerPid"]
print(json.dumps(response_as_json, indent=2))


## 3. Start Transfer (Provider -> Consumer)
The Provider calls `/dsp/current/transfers/rpc/setup-start`. The Provider dataplane begins pushing data to the Consumer's designated sink.
The transfer process moves from `REQUESTED` to `STARTED`.

In [ ]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


## 4. Suspend Transfer (Provider -> Consumer)
The Provider temporarily suspends data delivery by sending a `RpcTransferSuspensionMessage` via `/dsp/current/transfers/rpc/setup-suspension`.
The transfer process transitions from `STARTED` to `SUSPENDED`.

In [ ]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferSuspensionMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"],
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


## 5. Resume Transfer (Provider -> Consumer)
The Provider resumes the suspended push transfer by sending a `RpcTransferStartMessage` to `/dsp/current/transfers/rpc/setup-start`.
The transfer process returns to `STARTED`.

In [ ]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


## 6. Complete Transfer (Provider -> Consumer)
When data transmission is complete, the Provider calls `/dsp/current/transfers/rpc/setup-completion` to send a `TransferCompletionMessage`.
The transfer process reaches the terminal state `COMPLETED`.

In [ ]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-completion"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferCompletionMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


## 7. Query Final Transfer Process State
Query the Provider's Transfer Agent Management API (`/api/v1/transfer-agent/transfer-processes`) to inspect the final recorded process state.

In [ ]:
url = data_space_provider + "/api/v1/transfer-agent/transfer-processes"
response = requests.request("GET", url, headers=json_header)
print("Processes in Provider:")
print(json.dumps(response.json(), indent=2))
